## Model Training

### PPG Preprocessing
Three steps: mean removal, 4th-order Butterworth band-pass (0.5-5 Hz), and normalization by maximum absolute value.

### Environment setup
Run this first. The same notebook runs in two places:

- **Colab (training):** installs neurokit2, downloads the dataset from UCI onto the runtime (~3.3 GB, a few minutes; skipped if already there this session), mounts Google Drive, trains on the GPU and saves the model to `MyDrive/bp_results/<MODEL_NAME>.pt`. The runtime's disk is wiped when it disconnects, so the model is written to Drive every time validation improves.
- **VS Code / local CPU (everything else):** uses the local `data/cuff_less_bp_estimation` folder and **skips training**. Download the model file (e.g. `resnet_bp.pt`) from `MyDrive/bp_results` into `notebooks/resnet_models/` and the training cell loads it instead. Set `TRAIN = True` below to train locally anyway (slow on CPU).

In [ ]:
import os
import sys
import glob
import subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# train on Colab's GPU; locally, load the model trained there
TRAIN = IN_COLAB

UCI_URL = 'https://archive.ics.uci.edu/static/public/340/cuff+less+blood+pressure+estimation.zip'

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neurokit2'], check=True)

    # download straight from UCI onto the runtime (faster than copying from Drive)
    DATA_DIR = '/content/data'
    if len(glob.glob(f'{DATA_DIR}/Part_*.mat')) < 4:
        os.makedirs(DATA_DIR, exist_ok=True)
        print('Downloading dataset from UCI (~3.3 GB)...')
        subprocess.run(['wget', '-q', '-O', '/content/bp.zip', UCI_URL], check=True)
        subprocess.run(['unzip', '-q', '-o', '-j', '/content/bp.zip', '*Part_*.mat',
                        '-x', '__MACOSX/*', '-d', DATA_DIR], check=True)
        os.remove('/content/bp.zip')

    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = '/content/drive/MyDrive/bp_results'
    except Exception as e:
        print(f'WARNING: Google Drive not mounted ({e}).')
        print('Results will be saved on the runtime only and are lost when it disconnects.')
        SAVE_DIR = '/content/results'
else:
    # find the project root whether the kernel starts in notebooks/ or the root itself
    ROOT = None
    for base in ('.', '..'):
        if os.path.isdir(os.path.join(base, 'data', 'cuff_less_bp_estimation')):
            ROOT = os.path.abspath(base)
            break
    if ROOT is None:
        raise FileNotFoundError('Could not find data/cuff_less_bp_estimation - run from the project root or notebooks/')
    DATA_DIR = os.path.join(ROOT, 'data', 'cuff_less_bp_estimation')
    SAVE_DIR = os.path.join(ROOT, 'notebooks', 'resnet_models')

parts = sorted(os.path.basename(p) for p in glob.glob(os.path.join(DATA_DIR, 'Part_*.mat')))
if len(parts) < 4:
    raise FileNotFoundError(f'Expected Part_1.mat - Part_4.mat in {DATA_DIR}, found {parts}')

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Colab: {IN_COLAB}, train: {TRAIN}')
print(f'Data:  {DATA_DIR} -> {parts}')
print(f'Save:  {SAVE_DIR}')

### Experiment settings
Switches for the accuracy experiments. The defaults are the **v1 baseline** (the paper's setup), so leaving them alone reproduces the first run exactly.

- Change one group of switches per run and give each run its own `MODEL_NAME`, so earlier models in Drive are not overwritten.
- To evaluate a model locally, set the switches to the ones it was trained with. The cell checks this against the model file and stops early if they differ.
- Changing `INPUT_CHANNELS` changes the dataset, so rerun from the dataset cell. Adding `'ecg'` also drops recordings whose ECG has missing values, so test sets differ slightly from the PPG-only runs.

In [ ]:
# v1 baseline (paper):      USE_GAP=False, KERNEL=3, INPUT_CHANNELS=('ppg',), LOSS='mse',   WEIGHT_DECAY=0.0
# v2 less overfitting:      USE_GAP=True,  KERNEL=3, INPUT_CHANNELS=('ppg',), LOSS='huber', WEIGHT_DECAY=1e-4
# v3 + whole-beat kernels:  USE_GAP=True,  KERNEL=7, INPUT_CHANNELS=('ppg',), LOSS='huber', WEIGHT_DECAY=1e-4
# v4 + more inputs:         USE_GAP=True,  KERNEL=7, INPUT_CHANNELS=('ppg', 'd1', 'd2', 'ecg'), LOSS='huber', WEIGHT_DECAY=1e-4

MODEL_NAME = 'resnet_bp_v2'      # checkpoint file: <SAVE_DIR>/<MODEL_NAME>.pt, e.g. 'resnet_bp_v2'
USE_GAP = True               # True: global average pooling before the dense head (~66k head weights instead of 5.1M)
KERNEL = 3                    # conv kernel size: 3 sees ~0.3 s, 7 sees about one heartbeat
INPUT_CHANNELS = ('ppg',)     # any of 'ppg', 'd1' (1st derivative), 'd2' (2nd derivative), 'ecg'
LOSS = 'huber'                  # 'mse' or 'huber'
WEIGHT_DECAY = 1e-4            # > 0 switches Adam to AdamW with this weight decay

CONFIG = {'use_gap': USE_GAP, 'kernel': KERNEL, 'input_channels': list(INPUT_CHANNELS),
          'loss': LOSS, 'weight_decay': WEIGHT_DECAY}
# checkpoints saved before these switches existed carry no config: they are the v1 baseline
BASELINE_CONFIG = {'use_gap': False, 'kernel': 3, 'input_channels': ['ppg'],
                   'loss': 'mse', 'weight_decay': 0.0}
ARCH_KEYS = ('use_gap', 'kernel', 'input_channels')   # what must match to load a model

if not set(INPUT_CHANNELS) <= {'ppg', 'd1', 'd2', 'ecg'}:
    raise ValueError(f'unknown channel in {INPUT_CHANNELS}')
if LOSS not in ('mse', 'huber'):
    raise ValueError(f'unknown LOSS {LOSS!r}')

MODEL_PATH = os.path.join(SAVE_DIR, f'{MODEL_NAME}.pt')
print(f'Model:  {MODEL_PATH} ({"found" if os.path.exists(MODEL_PATH) else "not there yet"})')
print(f'Config: {CONFIG}')

# local evaluation: catch mismatched switches now, before the slow dataset build
if not TRAIN and os.path.exists(MODEL_PATH):
    import torch
    saved_cfg = torch.load(MODEL_PATH, map_location='cpu', weights_only=False).get('config', BASELINE_CONFIG)
    mismatch = {k: (saved_cfg[k], CONFIG[k]) for k in ARCH_KEYS if saved_cfg[k] != CONFIG[k]}
    if mismatch:
        raise ValueError(f'{MODEL_NAME}.pt was trained with different switches (saved, current): {mismatch}')

In [1]:
import numpy as np
import h5py
from scipy.signal import find_peaks
import neurokit2 as nk
from matplotlib import pyplot as plt

FS = 125            # sampling rate (Hz)
WINDOW_SIZE = 625   # 5 seconds at 125 Hz


**1) Mean removal** - (`nk.signal_detrend`, order 0) subtract the signal mean to remove the constant offset.

**2) Filtering** - Butterworth band-pass, order 4, 0.5-5 Hz. NeuroKit2's `signal_filter` applies it with zero phase, so pulse peaks are not shifted in time.

**3) Normalization** - divide by the maximum absolute value so the signal lies in [-1, 1].

In [2]:
def remove_mean(ppg):
    # order=0 detrend fits and subtracts a constant, i.e. the mean
    return nk.signal_detrend(np.asarray(ppg, dtype=float), order=0)


def bandpass_filter(ppg, fs=FS, low=0.5, high=5.0, order=4):
    return nk.signal_filter(ppg, sampling_rate=fs, lowcut=low, highcut=high,
                            method='butterworth', order=order)


def normalize(ppg):
    peak = np.max(np.abs(ppg))
    return ppg / peak if peak > 0 else ppg


def preprocess_ppg(ppg, fs=FS):
    """Mean removal -> Butterworth band-pass -> max-abs normalization."""
    return normalize(bandpass_filter(remove_mean(ppg), fs))


### Labels and window quality filter
SBP/DBP labels come from the ABP window (median of beat peaks/valleys, same rule as `data_exploration.ipynb`). A window is kept only if it has a valid label and a usable PPG.

In [3]:
def get_bp_labels(abp_window, fs=FS):
    """(SBP, DBP) from one ABP window, or None if the window is unusable."""
    min_gap = fs * 0.4
    peaks, _ = find_peaks(abp_window, distance=min_gap)
    valleys, _ = find_peaks(-abp_window, distance=min_gap)
    if len(peaks) == 0 or len(valleys) == 0:
        return None

    sys_vals = abp_window[peaks]
    sys_vals = sys_vals[(sys_vals >= 70) & (sys_vals <= 220)]
    dia_vals = abp_window[valleys]
    dia_vals = dia_vals[(dia_vals >= 30) & (dia_vals <= 140)]
    if len(sys_vals) == 0 or len(dia_vals) == 0:
        return None

    sbp, dbp = np.median(sys_vals), np.median(dia_vals)
    if sbp <= dbp or sbp - dbp < 10:
        return None
    return sbp, dbp


def good_ppg_window(ppg_win, fs=FS, min_hr=48, max_hr=180):
    """Reject flat/non-finite windows and windows whose beat count is physiologically implausible."""
    if not np.isfinite(ppg_win).all() or np.std(ppg_win) < 1e-3:
        return False
    peaks, _ = find_peaks(ppg_win, distance=int(0.33 * fs), prominence=0.2)
    n_sec = len(ppg_win) / fs
    return min_hr * n_sec / 60 <= len(peaks) <= max_hr * n_sec / 60


### Build the dataset
Load all four parts, window each recording, filter, label. `recording_id` is unique across parts so the split can be done by recording.

Each window holds one row per entry in `INPUT_CHANNELS`, so `X` has shape (windows, channels, 625). The derivatives come from the filtered PPG; the ECG gets its own 0.5-40 Hz band-pass because its QRS spikes are much sharper than a pulse wave. Every channel is normalized per window. The quality filter always looks at the PPG.

In [ ]:
# several channels as float32 would need ~5 GB, so they are stored as float16;
# the training loop converts each batch back to float32
X_DTYPE = np.float32 if len(INPUT_CHANNELS) == 1 else np.float16
NEED_DERIV = 'd1' in INPUT_CHANNELS or 'd2' in INPUT_CHANNELS
NEED_ECG = 'ecg' in INPUT_CHANNELS

X, y, rec_ids = [], [], []
rec_counter = 0
n_total = 0

for part in range(1, 5):
    with h5py.File(f'{DATA_DIR}/Part_{part}.mat', 'r') as f:
        data = f[f'Part_{part}']
        for i in range(data.shape[0]):
            record = np.array(f[data[i, 0]]).T          # (3, n): PPG, ABP, ECG
            ppg, abp, ecg = record[0], record[1], record[2]
            if not (np.isfinite(ppg).all() and np.isfinite(abp).all()
                    and (not NEED_ECG or np.isfinite(ecg).all())):
                rec_counter += 1
                continue

            # mean removal + band-pass on the full recording, then window
            filtered = bandpass_filter(remove_mean(ppg))
            signals = {'ppg': filtered}
            if NEED_DERIV:
                signals['d1'] = np.gradient(filtered)
                signals['d2'] = np.gradient(signals['d1'])
            if NEED_ECG:
                signals['ecg'] = bandpass_filter(remove_mean(ecg), low=0.5, high=40.0)

            n_windows = len(filtered) // WINDOW_SIZE
            for j in range(n_windows):
                s, e = j * WINDOW_SIZE, (j + 1) * WINDOW_SIZE
                n_total += 1
                win = normalize(filtered[s:e])
                labels = get_bp_labels(abp[s:e])
                if labels is None or not good_ppg_window(win):
                    continue
                X.append(np.stack([normalize(signals[c][s:e]) for c in INPUT_CHANNELS]).astype(X_DTYPE))
                y.append(labels)
                rec_ids.append(rec_counter)
            rec_counter += 1
    print(f'Part_{part} done: {len(X):,} windows kept so far')

X = np.stack(X)                         # (N, channels, 625)
y = np.array(y, dtype=np.float32)       # columns: SBP, DBP
rec_ids = np.array(rec_ids)
print(f'Kept {len(X):,} / {n_total:,} windows ({len(X)/n_total*100:.1f}%) from {len(np.unique(rec_ids)):,} recordings')
print(X.shape, X.dtype, y.shape)

### Split by recording
Windows from the same recording are near-duplicates, so whole recordings go to one split (70/15/15) to avoid leakage.

The splits are kept as index arrays (`train_idx`, `val_idx`, `test_idx`) into `X` rather than copies, because a multi-channel `X` can take several GB.

In [ ]:
rng = np.random.default_rng(42)
unique_recs = rng.permutation(np.unique(rec_ids))
n_train = int(0.70 * len(unique_recs))
n_val = int(0.15 * len(unique_recs))
train_recs = unique_recs[:n_train]
val_recs = unique_recs[n_train:n_train + n_val]
test_recs = unique_recs[n_train + n_val:]

train_idx = np.flatnonzero(np.isin(rec_ids, train_recs))
val_idx = np.flatnonzero(np.isin(rec_ids, val_recs))
test_idx = np.flatnonzero(np.isin(rec_ids, test_recs))
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

# standardize targets using training statistics only
y_mean, y_std = y_train.mean(axis=0), y_train.std(axis=0)
y_s = (y - y_mean) / y_std

print(f'train {len(train_idx):,}, val {len(val_idx):,}, test {len(test_idx):,} windows of shape {X.shape[1:]}')
print(f'target mean (SBP, DBP): {y_mean}, std: {y_std}')

Quick visual check: the first 5 s of one recording before and after preprocessing.

In [ ]:
with h5py.File(f'{DATA_DIR}/Part_1.mat', 'r') as f:
    rec = np.array(f[f['Part_1'][0, 0]]).T
raw = rec[0][:WINDOW_SIZE]
processed = normalize(bandpass_filter(remove_mean(rec[0]))[:WINDOW_SIZE])   # same steps as the dataset loop
labels = get_bp_labels(rec[1][:WINDOW_SIZE])
t = np.arange(WINDOW_SIZE) / FS

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(t, raw); axes[0].set(ylabel='Raw PPG', title='Before preprocessing')
label_text = f'SBP {labels[0]:.0f}, DBP {labels[1]:.0f}' if labels else 'no valid label'
axes[1].plot(t, processed); axes[1].set(ylabel='Preprocessed', xlabel='Time (s)',
                                        title=f'After preprocessing ({label_text})')
plt.tight_layout()

## 1D ResNet (PyTorch)
Architecture follows the referenced paper (IEEE SPACE 2025, DOI 10.1109/SPACE65882.2025.11171111):

- Input: one 5 s PPG window, shape 1 x 625
- Three residual blocks with 32, 64 and 128 filters. Each block: two Conv1d (kernel 3, zero padding) + BatchNorm + ReLU, a skip connection, then max-pooling (/2)
- Flatten -> Dense 512 -> Dropout 0.5
- Output head

The paper classifies into 4 AHA categories with a softmax head. Here the head is **2 linear outputs (standardized SBP, DBP)** so the model does regression, and the AHA category is derived from the predicted values afterwards.

Note: the paper's 99% accuracy comes from a random 80/20 split of windows, so windows from the same patient end up in both train and test. This notebook splits by recording, so expect noticeably weaker numbers. They are a more honest estimate of performance on unseen people.

**Switches** (see Experiment settings): `KERNEL` sets the conv kernel size, `INPUT_CHANNELS` sets the number of input channels, and `USE_GAP=True` replaces `Flatten` with global average pooling, so the dense head gets 128 numbers instead of 9,984.

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, Subset, DataLoader

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 64     # paper: 64
EPOCHS = 70         # paper: 70 (early stopping below usually ends sooner)
LR = 1e-3
PATIENCE = 8        # epochs without val MAE improvement before stopping

# one dataset over all windows; the splits are index subsets of it, so X is never copied
full_ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y_s))
train_ds = Subset(full_ds, train_idx.tolist())
val_ds = Subset(full_ds, val_idx.tolist())
test_ds = Subset(full_ds, test_idx.tolist())

# num_workers=0: worker processes are unreliable inside Jupyter on Windows
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024)
test_loader = DataLoader(test_ds, batch_size=1024)
print(f'device: {DEVICE}, {len(train_loader):,} train batches per epoch')

In [ ]:
class ResidualBlock(nn.Module):
    """Two conv-BN-ReLU layers with a skip connection, then max-pool /2."""

    def __init__(self, in_ch, out_ch, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        # 1x1 conv projects the input when the channel count changes
        self.skip = (nn.Identity() if in_ch == out_ch else
                     nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch)))
        self.pool = nn.MaxPool1d(2)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + self.skip(x))
        return self.pool(out)


class PPGResNet(nn.Module):
    def __init__(self, input_len=WINDOW_SIZE, in_channels=1, filters=(32, 64, 128), kernel_size=3,
                 use_gap=False, n_outputs=2):
        super().__init__()
        blocks, in_ch = [], in_channels
        for f in filters:
            blocks.append(ResidualBlock(in_ch, f, kernel_size))
            in_ch = f
        self.features = nn.Sequential(*blocks)
        if use_gap:
            # average each channel over time: (B, 128, 78) -> (B, 128)
            flatten = [nn.AdaptiveAvgPool1d(1), nn.Flatten()]
            n_flat = in_ch
        else:
            flatten = [nn.Flatten()]
            n_flat = in_ch * (input_len // 2 ** len(filters))     # 625 -> 312 -> 156 -> 78
        self.head = nn.Sequential(
            *flatten,
            nn.Linear(n_flat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_outputs),
        )

    def forward(self, x):
        return self.head(self.features(x))


def build_model(cfg):
    return PPGResNet(in_channels=len(cfg['input_channels']), kernel_size=cfg['kernel'],
                     use_gap=cfg['use_gap'])


model = build_model(CONFIG).to(DEVICE)
print(model)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

### Training
Adam optimizer (as in the paper). The loss is computed on the standardized targets: MSE by default, replacing the paper's cross-entropy because this is regression, or Huber with `LOSS='huber'`, which punishes large misses less. `WEIGHT_DECAY > 0` switches to AdamW. The learning rate is halved when validation MAE plateaus, and the weights with the best validation MAE are kept.

The printed `val` loss is always MSE, so runs with different `LOSS` settings stay comparable.

When `TRAIN` is False (local CPU), this cell loads the model saved by the Colab run instead of training, and the cells below evaluate that model.

In [ ]:
def predict(model, loader):
    """Predictions and targets in mmHg, shape (N, 2)."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(DEVICE).float()).cpu().numpy())
            targets.append(yb.numpy())
    return (np.concatenate(preds) * y_std + y_mean,
            np.concatenate(targets) * y_std + y_mean)


def save_checkpoint(state):
    # y_mean / y_std turn the model's standardized outputs back into mmHg; config rebuilds the architecture
    torch.save({'model': state, 'y_mean': y_mean, 'y_std': y_std, 'history': history,
                'config': CONFIG}, MODEL_PATH)


if TRAIN:
    criterion = nn.HuberLoss(delta=1.0) if LOSS == 'huber' else nn.MSELoss()
    if WEIGHT_DECAY > 0:
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    history = {'train_loss': [], 'val_loss': [], 'val_mae_sbp': [], 'val_mae_dbp': []}
    best_mae, best_state, bad_epochs = np.inf, None, 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE).float(), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)
        train_loss = running / len(train_ds)

        val_pred, val_true = predict(model, val_loader)
        val_loss = np.mean(((val_pred - val_true) / y_std) ** 2)     # always MSE, comparable across LOSS settings
        val_mae = np.abs(val_pred - val_true).mean(axis=0)
        scheduler.step(val_mae.mean())

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae_sbp'].append(val_mae[0])
        history['val_mae_dbp'].append(val_mae[1])
        print(f'epoch {epoch:2d} | train {train_loss:.4f} | val {val_loss:.4f} | '
              f'val MAE SBP {val_mae[0]:.2f}, DBP {val_mae[1]:.2f} mmHg | '
              f'lr {optimizer.param_groups[0]["lr"]:.1e} | {time.time() - t0:.0f}s')

        if val_mae.mean() < best_mae:
            best_mae, bad_epochs = val_mae.mean(), 0
            # CPU copy so the checkpoint loads on machines without a GPU
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            # save every improvement so a Colab disconnect doesn't lose the run
            save_checkpoint(best_state)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f'early stopping, best mean val MAE {best_mae:.2f} mmHg')
                break

    model.load_state_dict(best_state)
    save_checkpoint(best_state)
    print(f'saved best model to {MODEL_PATH}')
else:
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f'{MODEL_PATH} not found. Train on Colab first, then download '
                                f'{MODEL_NAME}.pt from MyDrive/bp_results into {SAVE_DIR}')
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    history = checkpoint.get('history')
    print(f'loaded trained model from {MODEL_PATH}')

In [ ]:
if history is None:
    print('no training history saved in this checkpoint')
else:
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'], label='validation')
    axes[0].set(xlabel='Epoch', ylabel='MSE (standardized)', title='Loss')
    axes[0].legend()
    axes[1].plot(epochs, history['val_mae_sbp'], label='SBP')
    axes[1].plot(epochs, history['val_mae_dbp'], label='DBP')
    axes[1].set(xlabel='Epoch', ylabel='MAE (mmHg)', title='Validation MAE')
    axes[1].legend()
    plt.tight_layout()

### Test set evaluation
MAE, RMSE and mean error ± SD for SBP and DBP, compared with a naive baseline that always predicts the training-set mean.

In [ ]:
test_pred, test_true = predict(model, test_loader)
baseline_pred = np.broadcast_to(y_mean, test_true.shape)

def report(name, pred, true):
    err = pred - true
    for k, target in enumerate(['SBP', 'DBP']):
        print(f'{name:9s} {target}: MAE {np.abs(err[:, k]).mean():5.2f} | '
              f'RMSE {np.sqrt((err[:, k] ** 2).mean()):5.2f} | '
              f'ME {err[:, k].mean():+5.2f} ± {err[:, k].std():5.2f} mmHg')

report('Baseline', baseline_pred, test_true)
report('ResNet', test_pred, test_true)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for k, (ax, target) in enumerate(zip(axes, ['SBP', 'DBP'])):
    ax.scatter(test_true[:, k], test_pred[:, k], s=2, alpha=0.1)
    lims = [test_true[:, k].min(), test_true[:, k].max()]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set(xlabel=f'True {target} (mmHg)', ylabel=f'Predicted {target} (mmHg)', title=target)
plt.tight_layout()

### AHA category from predicted BP
Map true and predicted (SBP, DBP) to the four AHA classes used in the paper (Normal, Elevated, Hypertension stage 1, Hypertension stage 2) and score the result as a classification.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

AHA_CLASSES = ['Normal', 'Elevated', 'HTN stage 1', 'HTN stage 2']

def aha_category(sbp, dbp):
    """0 Normal (<120 and <80), 1 Elevated (120-129 and <80),
    2 Stage 1 (130-139 or 80-89), 3 Stage 2 (>=140 or >=90)."""
    return np.select(
        [(sbp >= 140) | (dbp >= 90), (sbp >= 130) | (dbp >= 80), sbp >= 120],
        [3, 2, 1], default=0)

true_cls = aha_category(test_true[:, 0], test_true[:, 1])
pred_cls = aha_category(test_pred[:, 0], test_pred[:, 1])
print(classification_report(true_cls, pred_cls, labels=range(4), target_names=AHA_CLASSES, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(true_cls, pred_cls, labels=range(4), display_labels=AHA_CLASSES,
                                        normalize='true', values_format='.2f', cmap='Blues')
plt.title('AHA category (rows normalized)')
plt.tight_layout()

### Load a saved model
Works on any machine, CPU included, as long as the `ResidualBlock` / `PPGResNet` cells above have been run. `build_model` rebuilds the architecture from the config stored in the file (older files without one are the v1 baseline). `MODEL_PATH` comes from the experiment settings cell; change it if your file is somewhere else.

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
loaded = build_model(checkpoint.get('config', BASELINE_CONFIG))
loaded.load_state_dict(checkpoint['model'])
loaded.eval()
ckpt_mean, ckpt_std = checkpoint['y_mean'], checkpoint['y_std']

with torch.no_grad():
    out = loaded(torch.from_numpy(X[test_idx[:5]]).float()).numpy() * ckpt_std + ckpt_mean
for (sbp, dbp), (t_sbp, t_dbp) in zip(out, y_test[:5]):
    print(f'predicted {sbp:5.1f}/{dbp:4.1f}  true {t_sbp:5.1f}/{t_dbp:4.1f} mmHg')